# Japan Housing Market — Regional Price Map

This notebook creates an interactive map of Japan showing property 
price patterns across all prefectures.

Unlike static charts, the interactive map allows users to explore 
regional differences visually — hovering over prefectures reveals 
median prices, transaction volumes, and price trends.

**Tools:** Folium (Leaflet.js) for interactive mapping

## 1. Setup & Data Loading

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import json
import requests
import os

os.makedirs('visuals', exist_ok=True)

df = pd.read_csv('data/df_clean.csv')

print(f"Dataset loaded: {len(df):,} rows")
print(f"Prefectures: {df['Prefecture'].nunique()}")

Dataset loaded: 1,316,373 rows
Prefectures: 47


## 2. Preparing Prefecture-level Statistics

We calculate key metrics for each prefecture:
- Median transaction price
- Average price per m²
- Total number of transactions
- Price change over time (2015 vs 2023)

In [7]:
# Overall stats by prefecture
pref_stats = df.groupby('Prefecture').agg(
    median_price=('TotalTransactionValue', 'median'),
    avg_price_sqm=('PricePerSqM', 'mean'),
    total_transactions=('TotalTransactionValue', 'count')
).reset_index()

# Price change: 2015 vs 2023
price_2015 = (df[df['Year'] == 2015]
              .groupby('Prefecture')['TotalTransactionValue']
              .median())
price_2023 = (df[df['Year'] == 2023]
              .groupby('Prefecture')['TotalTransactionValue']
              .median())

price_change = ((price_2023 - price_2015) / price_2015 * 100).round(1)
price_change.name = 'price_change_pct'

pref_stats = pref_stats.merge(
    price_change, on='Prefecture', how='left'
)

pref_stats['median_price_M'] = (pref_stats['median_price'] / 1_000_000).round(1)
pref_stats['avg_price_sqm_K'] = (pref_stats['avg_price_sqm'] / 1_000).round(1)

print(f"Prefecture stats prepared: {len(pref_stats)} prefectures")
print("\nTop 5 by median price:")
print(pref_stats.nlargest(5, 'median_price')[
    ['Prefecture', 'median_price_M', 'price_change_pct', 'total_transactions']
].to_string(index=False))

Prefecture stats prepared: 47 prefectures

Top 5 by median price:
         Prefecture  median_price_M  price_change_pct  total_transactions
              Tokyo            44.0              28.6              135949
 Okinawa Prefecture            37.0              53.1                 519
Kanagawa Prefecture            36.0              20.6              123683
   Aichi Prefecture            32.0               6.2               80498
 Saitama Prefecture            28.0              18.5              101602


In [9]:
# Download Japan prefecture GeoJSON
url = 'https://raw.githubusercontent.com/dataofjapan/land/master/japan.geojson'
response = requests.get(url)

with open('data/japan_prefectures.geojson', 'wb') as f:
    f.write(response.content)

with open('data/japan_prefectures.geojson') as f:
    japan_geo = json.load(f)

# Check prefecture names in GeoJSON
geo_names = [f['properties']['nam_ja'] for f in japan_geo['features']]
print(f"GeoJSON loaded: {len(geo_names)} prefectures")
print("\nSample names from GeoJSON:")
print(geo_names[:5])

GeoJSON loaded: 47 prefectures

Sample names from GeoJSON:
['京都府', '佐賀県', '熊本県', '香川県', '愛知県']


In [11]:
# Map Japanese prefecture names to English
jp_to_en = {
    '北海道': 'Hokkaido',
    '青森県': 'Aomori Prefecture',
    '岩手県': 'Iwate Prefecture',
    '宮城県': 'Miyagi Prefecture',
    '秋田県': 'Akita Prefecture',
    '山形県': 'Yamagata Prefecture',
    '福島県': 'Fukushima Prefecture',
    '茨城県': 'Ibaraki Prefecture',
    '栃木県': 'Tochigi Prefecture',
    '群馬県': 'Gunma Prefecture',
    '埼玉県': 'Saitama Prefecture',
    '千葉県': 'Chiba Prefecture',
    '東京都': 'Tokyo',
    '神奈川県': 'Kanagawa Prefecture',
    '新潟県': 'Niigata Prefecture',
    '富山県': 'Toyama Prefecture',
    '石川県': 'Ishikawa Prefecture',
    '福井県': 'Fukui Prefecture',
    '山梨県': 'Yamanashi Prefecture',
    '長野県': 'Nagano Prefecture',
    '岐阜県': 'Gifu Prefecture',
    '静岡県': 'Shizuoka Prefecture',
    '愛知県': 'Aichi Prefecture',
    '三重県': 'Mie Prefecture',
    '滋賀県': 'Shiga Prefecture',
    '京都府': 'Kyoto Prefecture',
    '大阪府': 'Osaka Prefecture',
    '兵庫県': 'Hyogo Prefecture',
    '奈良県': 'Nara Prefecture',
    '和歌山県': 'Wakayama Prefecture',
    '鳥取県': 'Tottori Prefecture',
    '島根県': 'Shimane Prefecture',
    '岡山県': 'Okayama Prefecture',
    '広島県': 'Hiroshima Prefecture',
    '山口県': 'Yamaguchi Prefecture',
    '徳島県': 'Tokushima Prefecture',
    '香川県': 'Kagawa Prefecture',
    '愛媛県': 'Ehime Prefecture',
    '高知県': 'Kochi Prefecture',
    '福岡県': 'Fukuoka Prefecture',
    '佐賀県': 'Saga Prefecture',
    '長崎県': 'Nagasaki Prefecture',
    '熊本県': 'Kumamoto Prefecture',
    '大分県': 'Oita Prefecture',
    '宮崎県': 'Miyazaki Prefecture',
    '鹿児島県': 'Kagoshima Prefecture',
    '沖縄県': 'Okinawa Prefecture'
}

# Add English names to GeoJSON
for feature in japan_geo['features']:
    jp_name = feature['properties']['nam_ja']
    feature['properties']['name_en'] = jp_to_en.get(jp_name, jp_name)

# Merge with our stats
pref_stats_map = pref_stats.copy()

matched = sum(1 for name in pref_stats_map['Prefecture'] 
              if name in jp_to_en.values())
print(f"Matched prefectures: {matched}/47")

Matched prefectures: 47/47


## 4. Building the Interactive Map

In [14]:
# Create base map centered on Japan
m = folium.Map(
    location=[37.0, 137.0],
    zoom_start=5,
    tiles='CartoDB positron'
)

# Create choropleth layer — median price
choropleth = folium.Choropleth(
    geo_data=japan_geo,
    data=pref_stats_map,
    columns=['Prefecture', 'median_price_M'],
    key_on='feature.properties.name_en',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.5,
    legend_name='Median Property Price (¥ millions)',
    highlight=True
).add_to(m)

# Add tooltips with detailed info
for feature in japan_geo['features']:
    name_en = feature['properties']['name_en']
    row = pref_stats_map[pref_stats_map['Prefecture'] == name_en]
    
    if len(row) > 0:
        row = row.iloc[0]
        tooltip_text = f"""
        <b>{name_en}</b><br>
        Median Price: ¥{row['median_price_M']}M<br>
        Price/m²: ¥{row['avg_price_sqm_K']}K<br>
        Price Change (2015–2023): {row['price_change_pct']}%<br>
        Transactions: {row['total_transactions']:,}
        """
        folium.GeoJson(
            feature,
            style_function=lambda x: {
                'fillOpacity': 0,
                'weight': 0
            },
            tooltip=folium.Tooltip(tooltip_text)
        ).add_to(m)

# Save map
m.save('visuals/japan_price_map.html')
print("Map saved: visuals/japan_price_map.html")
print("Open this file in your browser to see the interactive map!")

Map saved: visuals/japan_price_map.html
Open this file in your browser to see the interactive map!


In [18]:
from IPython.display import IFrame
IFrame('visuals/japan_price_map.html', width=900, height=500)

### Observations

- **Tokyo and surrounding prefectures** (Kanagawa, Saitama, Chiba) 
  form a clear high-price cluster in the center of Honshu
- **Osaka and Aichi** show elevated prices as Japan's second and 
  third largest economic centers
- **Northern Honshu and Hokkaido** show the lowest prices — 
  consistent with our undervalued properties analysis
- The map confirms a clear **center-periphery price gradient** — 
  prices decrease as distance from Tokyo increases

## 5. Conclusions

### Key Findings

1. **Clear geographic price gradient** — prices decrease as distance 
   from Tokyo increases, confirming location as the dominant price driver
2. **Tokyo cluster** — Tokyo, Kanagawa, and Saitama form a high-price 
   zone with median prices 2-3x the national average
3. **Secondary urban centers** — Osaka and Aichi show elevated prices 
   as Japan's second and third largest economic hubs
4. **Northern periphery** — Hokkaido and Tohoku prefectures show 
   the lowest prices, consistent with depopulation trends
5. **Interactive exploration** — the map allows investors and analysts 
   to quickly identify regional opportunities

### Limitations
- Map shows median prices — masking within-prefecture variation 
  (e.g. central Tokyo vs outer Tokyo are very different)
- District-level data would reveal more granular patterns

## 6. Saving Results

In [23]:
pref_stats_map.to_csv('data/prefecture_stats.csv', index=False)
print(f"Saved: data/prefecture_stats.csv ({len(pref_stats_map)} prefectures)")
print("\nAll outputs:")
print("  - visuals/japan_price_map.html  (interactive map)")
print("  - data/prefecture_stats.csv     (prefecture statistics)")

Saved: data/prefecture_stats.csv (47 prefectures)

All outputs:
  - visuals/japan_price_map.html  (interactive map)
  - data/prefecture_stats.csv     (prefecture statistics)
